# Distances for the precision of synthetic tables

**Contents**
1. Motivation
2. Load the report
3. Results: Real tables
4. Results: Neighbors and summaries
5. Results: Simulated faults
6. Conclusion

References


## 1. Motivation

`flat.ipynb` measures a simulator's precision: the share of synthetic tables that lie at least as close to real tables as the real table they were drawn for.<br>
Closeness needs a distance between two tables, each summarized by up to 76 statistics, of which a table may lack whole groups.<br>
A calibrated simulator scores 0.5 under any distance, but how far a faulty one falls below it depends on the distance.<br>
We compare four distances and choose one for the required precision of 0.25.

### 1.1 Setup
- **Real tables:** the flat tables of `mixedbench` 1.0.15 whose sizes the default simulator draws, as in `flat.ipynb`, in a certifying and a fitting half of their sources.
- **Simulators:** each draws 39 synthetic tables per real table, with its sizes and column types: the default simulator, the independent one (the negative control) and TabICL's prior.
- **Toy:** 375 simulated real tables, each from a source of its own, with 77 statistics. Every synthetic table copies 16 of them, as it copies a real table's sizes. 60 come in five groups that a table shows or lacks as a whole, one group heavy-tailed. On the last one, 14% of the real tables tie at its edge.
- **Faults:** a calibrated toy simulator draws from the real tables' distribution, and a faulty one changes one thing: the spread, a share of the correlations, or the tied statistic. Each case runs on 10 seeds.

### 1.2 Measures
A table's reach is its mean distance to its 5 nearest real tables from other sources.<br>
The precision is the share of synthetic tables whose reach is at most their real table's, where a tie counts half.<br>
A distance first codes each statistic, then combines the gaps:
- **A, pooled ranks:** each statistic ranked among the real and the synthetic tables together, the mean absolute difference.
- **B, ranks among real tables:** each statistic ranked among the real tables from other sources, the mean absolute difference.
- **C, groups alike:** B, with each group of statistics weighted alike (the sizes, each column type's shape, the associations, the response).
- **D, robust scale:** each statistic centered on the median and scaled by the interquartile range of the real tables from other sources, the Euclidean distance with groups weighted alike (Zhao et al., 2026).

A statistic that only one of two tables shows costs 1 (under D, the real tables' range on it), and one that neither shows costs 0.<br>
We fixed the checks before the run:
- **One scale:** how far a real table's rank moves between the default simulator's pool and TabICL's. Every simulator should be measured on the same scale.
- **Control:** the gap between the precision of the default and the independent simulator, in standard errors over resampled sources. The two differ in the associations, so without those statistics the gap should vanish.
- **Concentration:** the share of the synthetic tables' excess reach over their real table's that the three largest statistics carry.
- **Faults:** the change in the toy's precision under each fault.

The default simulator's own precision is no criterion.

## 2. Load the report

In [1]:
import pandas as pd
from IPython.display import HTML, display

CANDIDATES = {
    'A': 'A, pooled ranks',
    'B': 'B, ranks among real tables',
    'C': 'C, groups alike',
    'D': 'D, robust scale',
}
SIMULATORS = {'default': 'default', 'independent': 'independent', 'tabicl': 'TabICL'}
HALVES = ['certifying', 'fitting']
REACHES = ['mean', 'median', 'largest']  # of the distances to the nearest real tables

report = pd.read_csv('distance-report.csv')
real = report[report['data'] != 'toy']
toy = report[report['data'] == 'toy']


def showTable(title: str, table: pd.DataFrame) -> None:
    # a table under a bold title, without a corner label
    table = table.rename_axis(index=None, columns=None)
    display(HTML(f'<b>{title}</b>' + table.to_html(escape=False)))


def readChosen(statistic: str, arm: str) -> pd.DataFrame:
    # a statistic at 5 neighbors and their mean distance, or one without neighbors, by half
    rows = real[(real['statistic'] == statistic) & (real['arm'] == arm)]
    chosen = rows['neighbors'].isna() | ((rows['neighbors'] == 5) & (rows['reach'] == 'mean'))
    return rows[chosen].pivot_table(index='data', columns='candidate', values='value')


tables = pd.read_csv('flat-real.csv')
inside = tables[tables['stratum'] == 'inside']
print(
    ', '.join(f'{(inside["part"] == half).sum()} real tables in the {half} half' for half in HALVES)
)

347 real tables in the certifying half, 424 real tables in the fitting half


## 3. Results: Real tables

We ask whether each distance measures every simulator on one scale and tells the negative control apart through the associations alone.

### Tables
In the first table, each row is a check and each column a distance, with the certifying half's value before the fitting half's.<br>
The precision uses 5 neighbors and the mean of their distances.<br>
Where a check prefers a value, the best distance on the certifying half is italicized.<br>
The second table gives how many of a real table's 5 nearest real tables two distances share.

In [2]:
# B, C and D rank the real tables among themselves, the same for every simulator
moved = readChosen('pooled_rank_moved', 'default, tabicl').reindex(
    columns=list(CANDIDATES), fill_value=0.0
)
CHECKS = {
    'precision, default': (readChosen('precision', 'default'), None),
    'precision, independent': (readChosen('precision', 'independent'), None),
    'precision, TabICL': (readChosen('precision', 'tabicl'), None),
    'gap between default and independent, in standard errors': (
        readChosen('gap', 'default - independent'),
        'largest',
    ),
    'the same gap without the associations': (
        readChosen('gap_without_relations', 'default - independent'),
        'smallest',
    ),
    'excess reach on the top three statistics, in %': (
        100 * readChosen('top_three_share', 'default'),
        None,
    ),
    "a real table's rank, moved between two simulators": (moved, 'smallest'),
}

rows = {}
for check, (values, best) in CHECKS.items():
    certifying = values.loc['certifying']
    if best == 'largest':
        favored = certifying[certifying == certifying.max()].index
    elif best == 'smallest':
        favored = certifying[certifying == certifying.min()].index
    else:
        favored = []
    digits = 0 if check.endswith('%') else 2
    cells = {}
    for candidate, label in CANDIDATES.items():
        halves = values.loc[HALVES, candidate]
        # a share of a negative excess says nothing
        if digits == 0 and halves.min() < 0:
            cell = 'negative excess'
        else:
            cell = ' / '.join(f'{value:.{digits}f}' for value in halves)
        cells[label] = f'<i>{cell}</i>' if candidate in favored else cell
    rows[check] = cells
showTable('Checks on the real tables, certifying / fitting half', pd.DataFrame(rows).T)

shared = real[real['statistic'] == 'shared_neighbors']
shared = shared.pivot_table(index='candidate', columns='data', values='value')
shared = (
    shared[HALVES].round(1).astype(str).agg(' / '.join, axis=1).to_frame('certifying / fitting')
)
showTable('Nearest real tables shared by two distances, of 5', shared)

,"A, pooled ranks","B, ranks among real tables","C, groups alike","D, robust scale"
"precision, default",0.15 / 0.15,0.39 / 0.35,0.39 / 0.34,0.54 / 0.48
"precision, independent",0.07 / 0.07,0.27 / 0.23,0.26 / 0.23,0.52 / 0.46
"precision, TabICL",0.44 / 0.43,0.48 / 0.48,0.47 / 0.47,0.48 / 0.48
"gap between default and independent, in standard errors",5.68 / 6.99,5.74 / 5.99,6.89 / 7.00,3.94 / 5.93
the same gap without the associations,3.92 / 3.60,2.45 / 0.56,0.32 / 0.60,1.43 / 1.23
"excess reach on the top three statistics, in %",46 / 45,41 / 40,42 / 43,negative excess
"a real table's rank, moved between two simulators",0.12 / 0.11,0.00 / 0.00,0.00 / 0.00,0.00 / 0.00


,certifying / fitting
"A, B",4.0 / 4.0
"A, C",3.5 / 3.5
"A, D",2.1 / 2.2
"B, C",3.9 / 3.9
"B, D",2.2 / 2.2
"C, D",2.0 / 2.0


Under A, a real table's rank moves by 0.12 between the default simulator's pool and TabICL's, so each simulator is measured on a scale of its own.<br>
A, B and C separate the independent simulator from the default by 5.7 to 7.0 standard errors, D by 3.9 and 5.9.<br>
Without the associations, the gap shrinks to 0.3 and 0.6 standard errors under C and to 2.5 and 0.6 under B, but stays at 3.9 and 3.6 under A.<br>
Under D, the real tables lie farther than their synthetic tables on the condition index, whose scale is unbounded, so the excess reach is negative and one statistic decides the distance.<br>
A and B share 4 of a real table's 5 nearest real tables, so they differ in the size of the gaps rather than in the neighbors.

## 4. Results: Neighbors and summaries

We ask whether the number of neighbors and the summary of their distances change the order of the simulators or their reaction to a fault, under B.

### Tables
In the first table, each row is a number of neighbors and each column a summary of their distances, with the precision of the default, the independent simulator and TabICL on the certifying half.<br>
In the second, each column is a fault of the toy and each cell its change in precision against the calibrated case under the mean, the median and the largest distance, averaged over seeds.

In [3]:
grid = real[(real['statistic'] == 'precision') & (real['candidate'] == 'B')]
grid = grid[grid['data'] == 'certifying']
grid = grid.pivot_table(index='neighbors', columns=['reach', 'arm'], values='value')
rows = {}
for neighbors in grid.index:
    cells = {}
    for reach in REACHES:
        cells[reach] = ' / '.join(f'{grid.loc[neighbors, (reach, arm)]:.2f}' for arm in SIMULATORS)
    rows[int(neighbors)] = cells
showTable('Precision under B: default / independent / TabICL', pd.DataFrame(rows).T)
gaps = real[(real['statistic'] == 'gap') & (real['candidate'] == 'B')]
print(
    f'The smallest gap between default and independent is {gaps["value"].min():.1f} standard errors.'
)

FAULTS = {
    'spread 1.25': 'spread 1.25 times',
    'independent 0.3': '30% of the associations weakened',
    'piled': 'one statistic piled at its edge',
}
seeds = toy[toy['candidate'] == 'B'].pivot_table(
    index=['neighbors', 'reach', 'seed'], columns='arm', values='value'
)
rows = {}
for neighbors in grid.index:
    cells = {}
    for fault, label in FAULTS.items():
        changes = []
        for reach in REACHES:
            paired = (
                seeds.loc[(neighbors, reach), fault] - seeds.loc[(neighbors, reach), 'calibrated']
            )
            changes.append(f'{paired.mean():+.2f}')
        cells[label] = ' / '.join(changes)
    rows[int(neighbors)] = cells
showTable("Change in the toy's precision under B: mean / median / largest", pd.DataFrame(rows).T)

,mean,median,largest
1,0.43 / 0.34 / 0.48,0.43 / 0.34 / 0.48,0.43 / 0.34 / 0.48
3,0.40 / 0.29 / 0.49,0.41 / 0.29 / 0.49,0.41 / 0.28 / 0.50
5,0.39 / 0.27 / 0.48,0.41 / 0.28 / 0.50,0.40 / 0.26 / 0.49
10,0.38 / 0.23 / 0.49,0.39 / 0.25 / 0.49,0.41 / 0.25 / 0.50
20,0.38 / 0.22 / 0.49,0.40 / 0.25 / 0.50,0.43 / 0.26 / 0.50
50,0.42 / 0.25 / 0.50,0.45 / 0.27 / 0.51,0.51 / 0.33 / 0.52


The smallest gap between default and independent is 4.8 standard errors.


,spread 1.25 times,30% of the associations weakened,one statistic piled at its edge
1,-0.12 / -0.12 / -0.12,-0.14 / -0.14 / -0.14,-0.01 / -0.01 / -0.01
3,-0.17 / -0.15 / -0.17,-0.16 / -0.14 / -0.12,-0.02 / -0.01 / -0.02
5,-0.19 / -0.17 / -0.18,-0.15 / -0.12 / -0.11,-0.02 / -0.02 / -0.02
10,-0.22 / -0.19 / -0.20,-0.13 / -0.11 / -0.08,-0.02 / -0.02 / -0.02
20,-0.23 / -0.21 / -0.20,-0.10 / -0.08 / -0.03,-0.02 / -0.02 / -0.02
50,-0.23 / -0.20 / -0.20,-0.05 / -0.02 / -0.05,-0.02 / -0.02 / -0.02


At every number of neighbors and under every summary, the independent simulator scores below the default and the default below TabICL.<br>
The default's precision moves between 0.34 and 0.51 over the grid and the two halves.<br>
On the toy, more neighbors react more to a wider simulator (from -0.12 at 1 neighbor to -0.23 at 50) and less to weakened associations (from -0.14 to -0.05), and 5 neighbors react to both.<br>
The mean reacts at least as much as the median and the largest distance in every cell.

## 5. Results: Simulated faults

We ask how each distance reacts to a known fault of the simulator.

### Table
Each row is a case of the toy and each column a distance, at 5 neighbors and the mean of their distances.<br>
The first row gives the calibrated case's precision, the others its change under each fault, paired by seed and averaged over 10 seeds.

In [4]:
CASES = {
    'spread 0.8': 'spread 0.8 times',
    'spread 1.1': 'spread 1.1 times',
    'spread 1.25': 'spread 1.25 times',
    'spread 1.5': 'spread 1.5 times',
    'independent 0.1': '10% of the associations weakened',
    'independent 0.3': '30% of the associations weakened',
    'piled': 'one statistic piled at its edge',
}
chosen = toy[(toy['neighbors'] == 5) & (toy['reach'] == 'mean')]
seeds = chosen.pivot_table(index=['arm', 'seed'], columns='candidate', values='value')
calibrated = seeds.loc['calibrated']
rows = {'calibrated': {CANDIDATES[c]: f'{calibrated[c].mean():.2f}' for c in CANDIDATES}}
errors = [calibrated.std() / len(calibrated) ** 0.5]
for case, label in CASES.items():
    change = seeds.loc[case] - calibrated
    errors.append(change.std() / len(change) ** 0.5)
    rows[label] = {CANDIDATES[c]: f'{change[c].mean():+.2f}' for c in CANDIDATES}
showTable('Precision on the toy', pd.DataFrame(rows).T)
print(f'The largest standard error over the seeds is {max(error.max() for error in errors):.3f}.')

,"A, pooled ranks","B, ranks among real tables","C, groups alike","D, robust scale"
calibrated,0.51,0.51,0.51,0.50
spread 0.8 times,+0.15,+0.18,+0.18,+0.19
spread 1.1 times,-0.09,-0.08,-0.09,-0.08
spread 1.25 times,-0.22,-0.19,-0.19,-0.19
spread 1.5 times,-0.37,-0.32,-0.32,-0.31
10% of the associations weakened,-0.05,-0.05,-0.05,-0.02
30% of the associations weakened,-0.15,-0.15,-0.16,-0.04
one statistic piled at its edge,-0.07,-0.02,-0.02,-0.00


The largest standard error over the seeds is 0.010.


Every distance scores the calibrated toy at 0.50 or 0.51.<br>
A, B and C react alike to the spread and to weakened associations, and D reacts to 30% weakened associations by only -0.04.<br>
One statistic piled at its edge costs A 0.07, more than 10% weakened associations do, and B and C 0.02.<br>
B and C react less than A to a wider simulator (-0.19 against -0.22 at 1.25 times the spread), since a synthetic table beyond every real table is ranked like the most extreme one.

## 6. Conclusion

| distance | verdict |
|---|---|
| A, pooled ranks | rejected: a real table's rank moves with the simulator, and one piled statistic outweighs weakened associations |
| B, ranks among real tables | chosen: one scale for every simulator, a clear gap to the negative control, a reaction to every fault |
| C, groups alike | kept as a second reading: it scores like B, for a list of groups |
| D, robust scale | rejected: one unbounded statistic decides it, and it barely reacts to weakened associations |

The precision uses B with the mean distance to 5 neighbors.<br>
Up to 20 neighbors, the mean moves the default's precision by about 0.05, so the required 0.25 is not read more finely than that.

## References

Naeem, M. F., Oh, S. J., Uh, Y., Choi, Y., & Yoo, J. (2020). Reliable fidelity and diversity metrics for generative models. *Proceedings of the 37th International Conference on Machine Learning*, 7176-7185. https://arxiv.org/abs/2002.09797

Zhao, H., Thompson, R., Steinberg, D. M., Rahman, A., Bonilla, E. V., & Ong, C. S. (2026). From synthetic priors to model behavior: Structural coverage in tabular foundation models. *arXiv preprint*. https://arxiv.org/abs/2609.06912